In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("df_combined_AT.csv")
df.head()

,utc_timestamp,AT_load_actual_entsoe_transparency,temperature_2m (°C),rain (mm),relative_humidity_2m (%),sunshine_duration (s)
0,2015-01-01 00:00:00+00:00,5946.0,-7.0,0.0,94,0.0
1,2015-01-01 01:00:00+00:00,5726.0,-7.9,0.0,95,0.0
2,2015-01-01 02:00:00+00:00,5347.0,-7.8,0.0,94,0.0
3,2015-01-01 03:00:00+00:00,5249.0,-7.9,0.0,91,0.0
4,2015-01-01 04:00:00+00:00,5309.0,-8.6,0.0,92,0.0


In [3]:
len(df)

50400

In [12]:
# ubah index jadi timestamp

In [4]:
df_dummy = df.drop(columns=['utc_timestamp'])

In [5]:
df_dummy.shape[0]

50400

In [6]:
scaler = MinMaxScaler()
scaler.fit(df_dummy[:int(df_dummy.shape[0]*0.8)]) # train
scaled_data = scaler.transform(df_dummy)

In [7]:
import pickle

In [24]:
with open('scaler_fix.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [8]:
scaled_data.shape[0]*0.8

40320.0

In [11]:
scaled_data[int(scaled_data.shape[0]*0.2):][:,0].shape

(40320,)

In [12]:
scaled_data.shape

(50400, 5)

In [13]:
scaled_data[:, 0].shape

(50400,)

In [14]:
# Keep the entire dataset for X, and just isolate the target column (column 0) for y
scaled_data_x = scaled_data
scaled_data_y = scaled_data[:, 0]

print(f"scaled_data_x {scaled_data_x.shape}")
print(f"scaled_data_y {scaled_data_y.shape}")

scaled_data_x (50400, 5)
scaled_data_y (50400,)


In [17]:
df_dummy.columns

Index(['AT_load_actual_entsoe_transparency', 'temperature_2m (°C)',
       'rain (mm)', 'relative_humidity_2m (%)', 'sunshine_duration (s)'],
      dtype='object')

In [18]:
scaled_data_y.shape

(50400,)

In [23]:
np.save('x_scaled_fix.npy', scaled_data_x)
np.save('y_scaled_fix.npy', scaled_data_y)

In [19]:
scaled_data_y.shape

(50400,)

In [20]:
def create_supervised_data(scaled_data_x, scaled_data_y, lag=24, horizon=24):
    X, y = [], []
    
    # Loop over the number of timestamps (total rows)
    for i in range(lag, len(scaled_data_x) - horizon + 1):
        # Rows: from (i - lag) to i. Columns: all columns (features)
        X.append(scaled_data_x[i - lag:i, :]) 
        # Rows: from i to (i + horizon). Columns: all columns (target)
        y.append(scaled_data_y[i:i + horizon :]) 
    
    # for i in range(lag, len(scaled_data_x[0]) - horizon + 1):
    #     X.append(scaled_data_x[:, i - lag:i])
    #     y.append(scaled_data_y[:, i:i + horizon])
        
    return np.array(X), np.array(y)

def train_val_test_split(X, y, train_ratio=0.8, val_ratio=0.1):
    total = len(X)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    X_train, y_train = X[:train_end], y[:train_end]
    X_val, y_val = X[train_end:val_end], y[train_end:val_end]
    X_test, y_test = X[val_end:], y[val_end:]

    return X_train, y_train, X_val, y_val, X_test, y_test

In [21]:
X, y = create_supervised_data(scaled_data_x, scaled_data_y, lag=24, horizon=24)
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X, y, train_ratio=0.8, val_ratio=0.1)

print(f"X_train.shape: {X_train.shape}")
print(f"y_train.shape: {y_train.shape}")
print(f"X_val.shape: {X_val.shape}")
print(f"y_val.shape: {y_val.shape}")
print(f"X_test.shape: {X_test.shape}")
print(f"y_test.shape: {y_test.shape}")

X_train.shape: (40282, 24, 5)
y_train.shape: (40282, 24)
X_val.shape: (5035, 24, 5)
y_val.shape: (5035, 24)
X_test.shape: (5036, 24, 5)
y_test.shape: (5036, 24)


In [22]:
import os
import numpy as np

# Create the directory if it doesn't exist
os.makedirs('dataset_ready', exist_ok=True)

# Save each array to the folder
np.save('dataset_ready/X_train.npy', X_train)
np.save('dataset_ready/y_train.npy', y_train)
np.save('dataset_ready/X_val.npy', X_val)
np.save('dataset_ready/y_val.npy', y_val)
np.save('dataset_ready/X_test.npy', X_test)
np.save('dataset_ready/y_test.npy', y_test)

print("Data successfully saved to the 'dataset_ready' folder!")

Data successfully saved to the 'dataset_ready' folder!


# DRAFT

In [63]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [78]:
def get_model(num_d, seq_len):
    # input_layer = layers.Input(shape=(num_d, seq_len))  # shape: (batch, num_d, seq_len)
    input_layer = layers.Input(shape=(seq_len, num_d)) 
    # seq = [ 11 12 13 14 15 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 ]
    # num_d = 8; seq_len = 24
    outputs = []
    x = layers.LSTM(64, return_sequences=False)(input_layer)
    summed = layers.Dense(seq_len)(x)
    model = keras.models.Model(inputs=input_layer, outputs=summed, name="DLinearFunctional")
    model.compile(optimizer='adam', loss='mse')
    return model

In [80]:
num_d = X_train.shape[2] 
seq_len = X_train.shape[1]
batch = 32

In [82]:
def mean_absolute_percentage_error(y_true, y_pred):
    """
    Compute Mean Absolute Percentage Error (MAPE)

    Parameters:
    - y_true: array-like, true values
    - y_pred: array-like, predicted values

    Returns:
    - mape: float, MAPE in percentage
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Avoid division by zero
    non_zero = y_true != 0
    if not np.any(non_zero):
        return np.nan  # or raise an error

    mape = np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100
    return mape

In [83]:
# Training
print("\n====== TRAINING ======")
model = get_model(num_d, seq_len)
# model.fit(np.moveaxis(X_train, 1, 2), y_train, validation_data=(np.moveaxis(X_val, 1, 2), y_val), epochs=200, batch_size=batch)
model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=200, 
    batch_size=batch
)

print("\n====== TESTING ======")
pred = model.predict(np.moveaxis(X_test, 1, 2))
mape = mean_absolute_percentage_error(y_test[:,0], pred)
print("MAPE: ", mape)


====== TRAINING ======
Epoch 1/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 15s 12ms/step - loss: 0.0159 - val_loss: 0.0119
Epoch 2/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.0094 - val_loss: 0.0110
Epoch 3/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.0088 - val_loss: 0.0105
Epoch 4/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 0.0084 - val_loss: 0.0104
Epoch 5/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - loss: 0.0081 - val_loss: 0.0102
Epoch 6/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 0.0078 - val_loss: 0.0099
Epoch 7/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - loss: 0.0075 - val_loss: 0.0095
Epoch 8/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 0.0074 - val_loss: 0.0092
Epoch 9/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 0.0071 - val_loss: 0.0095
Epoch 10/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 0.0069 - val_loss: 0.0097
Epoch 11/200
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0

ValueError: Input 0 of layer "DLinearFunctional" is incompatible with the layer: expected shape=(None, 24, 5), found shape=(32, 5, 24)

In [85]:
print("\n====== TESTING ======")
pred = model.predict(X_test)
# Compare the full 24-step prediction to the full 24-step ground truth
mape = mean_absolute_percentage_error(y_test, pred)
print("MAPE: ", mape)


====== TESTING ======
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
MAPE:  11.192775304801145


In [87]:
y_test

array([[0.64049709, 0.65519282, 0.64355459, ..., 0.44077325, 0.49265214,
        0.57125949],
       [0.65519282, 0.64355459, 0.64661209, ..., 0.49265214, 0.57125949,
        0.63181773],
       [0.64355459, 0.64661209, 0.66466121, ..., 0.57125949, 0.63181773,
        0.64680935],
       ...,
       [0.48633988, 0.4596114 , 0.45014301, ..., 0.59147845, 0.55942401,
        0.51957787],
       [0.4596114 , 0.45014301, 0.44156228, ..., 0.55942401, 0.51957787,
        0.48959463],
       [0.45014301, 0.44156228, 0.45428543, ..., 0.51957787, 0.48959463,
        0.46661406]])

In [88]:
pred

array([[0.7184795 , 0.7579473 , 0.77290255, ..., 0.50078934, 0.54708403,
        0.60626906],
       [0.75668037, 0.7655729 , 0.7631825 , ..., 0.5415205 , 0.6071593 ,
        0.6628215 ],
       [0.73454833, 0.72950184, 0.7226118 , ..., 0.59799963, 0.6571791 ,
        0.69229656],
       ...,
       [0.5005288 , 0.47838685, 0.45655826, ..., 0.59018844, 0.5550054 ,
        0.51800936],
       [0.47965127, 0.45979205, 0.4453943 , ..., 0.55590516, 0.5203763 ,
        0.48501933],
       [0.45038074, 0.43320954, 0.4335341 , ..., 0.5189037 , 0.48423964,
        0.45183918]], dtype=float32)

In [89]:
# 1. Calculate row-wise MAPE for each of the 4028 test samples
row_mapes = []
for i in range(len(y_test)):
    # Calculate MAPE safely avoiding division by zero for each sample
    non_zero = y_test[i] != 0
    if np.any(non_zero):
        m = np.mean(np.abs((y_test[i][non_zero] - pred[i][non_zero]) / y_test[i][non_zero])) * 100
    else:
        m = np.nan
    row_mapes.append(m)

# 2. Get the correct timestamp indices
# X starts at index `lag` (24). Test set starts after train (80%) + val (10%) splits.
total_samples = len(X)
train_end = int(total_samples * 0.8)
val_end = train_end + int(total_samples * 0.1)

# Align with original dataframe timestamps
# Provide the timestamps that correspond to the START of the 24-step test forecast
test_timestamps = df['utc_timestamp'].iloc[24 + val_end : 24 + val_end + len(y_test)].values

# 3. Create the Results DataFrame
results_df = pd.DataFrame({
    'timestamp': test_timestamps,
    'y_test': list(y_test),        # Stores the array of 24 real values
    'pred': list(pred),            # Stores the array of 24 predicted values
    'MAPE': row_mapes              # Mape for this specific 24-step prediction
})

# Display the first few rows
results_df.head()

,timestamp,y_test,pred,MAPE
0,2019-02-20 05:00:00+00:00,"[0.6404970904428444, 0.6551928198047143, 0.643...","[0.7184795, 0.7579473, 0.77290255, 0.7729799, ...",14.378843
1,2019-02-20 06:00:00+00:00,"[0.6551928198047143, 0.6435545911825624, 0.646...","[0.75668037, 0.7655729, 0.7631825, 0.75368714,...",12.243619
2,2019-02-20 07:00:00+00:00,"[0.6435545911825624, 0.6466120919222802, 0.664...","[0.73454833, 0.72950184, 0.7226118, 0.71311504...",10.392519
3,2019-02-20 08:00:00+00:00,"[0.6466120919222802, 0.6646612091922279, 0.638...","[0.77052486, 0.76382935, 0.75992525, 0.7543682...",17.226431
4,2019-02-20 09:00:00+00:00,"[0.6646612091922279, 0.6384258802643259, 0.610...","[0.75376034, 0.7440375, 0.7308707, 0.7224833, ...",15.423283


In [91]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4028 entries, 0 to 4027
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  4028 non-null   object 
 1   y_test     4028 non-null   object 
 2   pred       4028 non-null   object 
 3   MAPE       4028 non-null   float64
dtypes: float64(1), object(3)
memory usage: 126.0+ KB


In [93]:
y_test.shape

(4028, 24)

In [94]:
y_train.shape

(32218, 24)

In [95]:
X_train.shape

(32218, 24, 5)

# univariate

In [ ]:
import os
import numpy as np

# 1. Isolate ONLY the load feature (assuming it's at index 0 based on your target column)
# We keep it as a 2D array (N, 1) so it works with create_supervised_data
scaled_data_x_uni = scaled_data[:, 0:1] 
scaled_data_y_uni = scaled_data[:, 0]

print(f"scaled_data_x_uni shape: {scaled_data_x_uni.shape}")
print(f"scaled_data_y_uni shape: {scaled_data_y_uni.shape}")

# 2. Create sliding windows for the univariate data
X_uni, y_uni = create_supervised_data(scaled_data_x_uni, scaled_data_y_uni, lag=24, horizon=24)

# 3. Split the data
X_train_uni, y_train_uni, X_val_uni, y_val_uni, X_test_uni, y_test_uni = train_val_test_split(
    X_uni, y_uni, train_ratio=0.8, val_ratio=0.1
)

print(f"X_train_uni shape: {X_train_uni.shape}")
print(f"y_train_uni shape: {y_train_uni.shape}")

# 4. Save to UNIVARIATE folder
os.makedirs('UNIVARIATE', exist_ok=True)
np.save('UNIVARIATE/X_train.npy', X_train_uni)
np.save('UNIVARIATE/y_train.npy', y_train_uni)
np.save('UNIVARIATE/X_val.npy', X_val_uni)
np.save('UNIVARIATE/y_val.npy', y_val_uni)
np.save('UNIVARIATE/X_test.npy', X_test_uni)
np.save('UNIVARIATE/y_test.npy', y_test_uni)

print("Univariate data successfully saved to the 'UNIVARIATE' folder!")

scaled_data_x_uni shape: (50400, 1)
scaled_data_y_uni shape: (50400,)
X_train_uni shape: (40282, 24, 1)
y_train_uni shape: (40282, 24)
Univariate data successfully saved to the 'UNIVARIATE' folder!


: 